In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hijest/genre-classification-dataset-imdb")

print("Path to dataset files:", path)

100%|██████████| 41.7M/41.7M [00:00<00:00, 98.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/hijest/genre-classification-dataset-imdb/versions/1


In [2]:
with open("train_data.txt", "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline())

1 ::: Oscar et la dame rose (2009) ::: drama ::: Listening in to a conversation between his doctor and parents, 10-year-old Oscar learns what nobody has the courage to tell him. He only has a few weeks to live. Furious, he refuses to speak to anyone except straight-talking Rose, the lady in pink he meets on the hospital stairs. As Christmas approaches, Rose uses her fantastical experiences as a professional wrestler, her imagination, wit and charm to allow Oscar to live life and love to the full, in the company of his friends Pop Corn, Einstein, Bacon and childhood sweetheart Peggy Blue.

2 ::: Cupid (1997) ::: thriller ::: A brother and sister with a past incestuous relationship have a current murderous relationship. He murders the women who reject him and she murders the women who get too close to him.

3 ::: Young, Wild and Wonderful (1980) ::: adult ::: As the bus empties the students for their field trip to the Museum of Natural History, little does the tour guide suspect that the

In [8]:
import pandas as pd

data = []

with open("train_data.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(" ::: ")

        if len(parts) >= 4:
            movie_id = parts[0]
            title = parts[1]
            genre = parts[2]
            description = parts[3]

            data.append([title, genre, description])

df = pd.DataFrame(
    data,
    columns=["title", "genre", "description"]
)

print(df.head())
print(df.shape)

                              title     genre  \
0      Oscar et la dame rose (2009)     drama   
1                      Cupid (1997)  thriller   
2  Young, Wild and Wonderful (1980)     adult   
3             The Secret Sin (1915)     drama   
4            The Unrecovered (2007)     drama   

                                         description  
0  Listening in to a conversation between his doc...  
1  A brother and sister with a past incestuous re...  
2  As the bus empties the students for their fiel...  
3  To help their unemployed father make ends meet...  
4  The film's title refers not only to the un-rec...  
(46511, 3)


# TFIDF + Logistic Regression


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df["description"]
y = df["genre"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

modell = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=10000
    )),
    ("clf", LogisticRegression(max_iter=1000))
])

modell.fit(X_train, y_train)

pred = modell.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.5777706116306568


In [17]:
sample_plot = """
A family moves into an old countryside mansion hoping for a fresh start.
Soon, strange noises, terrifying visions, and unexplained disappearances
reveal the house's dark and deadly past.
"""

prediction = modell.predict([sample_plot])

print("Predicted Genre:", prediction[0])

Predicted Genre: horror


# TFIDF + Random forestclassifier


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = df["description"]
y = df["genre"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=10000
    )),
    ("clf", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.48650972804471676
